# 00 · Roadmap & the NVIDIA edge-AI stack

**Goal of the project:** a small wheeled robot with a camera and a small arm. You say *"bring me the red cup from the kitchen"*, and it understands, drives to the kitchen, finds the right object among others, picks it up and brings it back. Everything runs **on the robot's own small NVIDIA GPU**, with no cloud.

To get there we need three things (your three pillars):

| Pillar | Question | Where in this repo |
|---|---|---|
| 1. An optimized model | How small can a vision-language model be and still ground language in pixels? | notebooks 01-05, `src/edge_vlm/model.py` |
| 2. Make it run on the hardware | ONNX, TensorRT, TensorRT Edge-LLM, containers, ROS 2 | notebooks 05-07, `deploy/` |
| 3. Respect hardware constraints | Memory, bandwidth, power, latency budgets | this notebook, `src/edge_vlm/memory.py` |

### Notebook map
| # | Notebook | Runs on |
|---|---|---|
| 00 | Roadmap & NVIDIA stack (this) | anything (CPU) |
| 01 | Build a VLM from scratch, piece by piece | anything (CPU) |
| 02 | The robot world & the imitation-learning dataset | anything (CPU) |
| 03 | Train the tiny robot VLM & evaluate it in closed loop | GPU recommended |
| 04 | Pretrained small VLMs (SmolVLM) + LoRA fine-tuning | **NVIDIA GPU** |
| 05 | Optimization: quantization, KV cache, ONNX export | GPU recommended |
| 06 | TensorRT & Jetson deployment | **Jetson** (reading works anywhere) |
| 07 | The full robot agent loop & the path to real hardware | anything |

## The system we are building

```
  voice ──► ASR (Riva / Whisper) ──► text command
                                        │
  camera ──► image ─────────────────────┤
  localization (cuVSLAM) ─► room ───────┤
  gripper sensor ─► holding? ───────────┤
                                        ▼
                     ┌────────────────────────────────┐
                     │  SYSTEM 2: small VLM  (~1-5 Hz)│   <- this repo's main focus
                     │  'what skill next?'            │
                     └───────────────┬────────────────┘
                                     │ e.g. {"skill": "pick", "object": "red cup"}
                              guardrail (validate + preconditions)
                                     ▼
                     ┌────────────────────────────────┐
                     │  SYSTEM 1: skills (20-1000 Hz) │
                     │  Nav2 path following, obstacle │
                     │  avoidance, grasping, motors   │
                     └────────────────────────────────┘
```

This **System 2 / System 1** split is the same idea NVIDIA uses in **Isaac GR00T** humanoid models: a slow VLM reasons about *what* to do, and fast controllers handle *how*. It is also what makes a tiny GPU workable. The VLM only has to answer a few times per second, never in the 100 Hz motor loop.

## The NVIDIA stack, layer by layer

| Layer | What it is | Why you care |
|---|---|---|
| **Jetson modules** | Orin Nano 8GB, Orin NX 16GB, AGX Orin 64GB, **Thor** (Blackwell) | The robot's computer. CPU and GPU **share** one LPDDR memory pool. |
| **JetPack** | Ubuntu-based OS (L4T) + CUDA + cuDNN + TensorRT + drivers | Pins your CUDA/TensorRT versions. JetPack 6 = Orin, JetPack 7 = Thor (+Orin support rolling in). Check the version you flash. |
| **TensorRT** | Optimizing compiler + runtime for NN graphs (ONNX in, `.engine` out) | Layer fusion, FP16/INT8 kernels, picks the fastest kernels *for your exact GPU*. |
| **TensorRT Edge-LLM** | C++ LLM/VLM runtime for Jetson & DRIVE ([github](https://github.com/NVIDIA/TensorRT-Edge-LLM)) | KV-cache management, INT4/FP8/NVFP4 quantized LLMs & VLMs, no Python at runtime. The production path for a pretrained VLM. |
| **TensorRT Model Optimizer** | Python library for quantization (AWQ, SmoothQuant, FP8, INT4), pruning, distillation | Produces checkpoints that TensorRT / Edge-LLM consume. |
| **llama.cpp / Ollama** | Community runtimes with GGUF quantization, CUDA backend | Fastest way to *try* a VLM on a Jetson (e.g. SmolVLM GGUF). |
| **jetson-containers / Jetson AI Lab** | Pre-built Docker images (PyTorch, llama.cpp, ROS...) + tutorials ([jetson-ai-lab.com](https://www.jetson-ai-lab.com)) | Saves days of building aarch64 wheels. |
| **ROS 2 + Isaac ROS** | GPU-accelerated ROS 2 packages: cuVSLAM (localization), nvblox (3D mapping), FoundationPose (object pose), cuMotion (arm planning) | System 1: navigation, perception, manipulation. |
| **Isaac Sim / Isaac Lab** | Photoreal physics simulator / robot-learning framework | Generate training data & test the robot before building it. |
| **Cosmos** | World foundation models, including **Cosmos Reason**, a physical-AI reasoning VLM | Strong reference VLMs for robotics, and synthetic data augmentation. |
| **Isaac GR00T** | Open robot foundation models (VLA) | Where the field is going: VLM + action head end-to-end. |

Versions move fast. Treat the names above as *what to look for* and check the current release notes before installing anything.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))  # or: pip install -e .

from edge_vlm.hardware import PROFILES, DEFAULT_TARGET
from edge_vlm.memory import format_table

rows = [{'key': p.key, 'name': p.name, 'memory_GB': p.memory_gb, 'usable_GB': p.usable_gb,
         'bandwidth_GB/s': p.memory_bandwidth_gbs, 'AI perf': p.ai_perf, 'power_W': p.power_modes_w}
        for p in PROFILES.values()]
print(format_table(rows))
print('\nDesign target:', DEFAULT_TARGET, '-> scale up later without changing code')

## Pillar 3 first: the two equations that decide everything

**1. Memory: does it fit?**

$$\text{total} \approx \underbrace{N_{params}\cdot\text{bytes/param}}_{\text{weights}} + \underbrace{2 \cdot L \cdot H_{kv} \cdot d_{head} \cdot T \cdot \text{bytes}}_{\text{KV cache}} + \text{activations/runtime}$$

On an 8 GB Orin Nano, the OS, ROS 2, camera and navigation stack also live in those 8 GB. We reserve ~2.5 GB for them.

**2. Speed: how many tokens per second?**

Generating one token at batch size 1 has to read *every* LM weight from DRAM once. Arithmetic is cheap and memory traffic is the bottleneck:

$$\text{tokens/s} \lesssim \frac{\text{memory bandwidth}}{\text{bytes of LM weights}}$$

So **quantizing from FP16 to INT4 is ~4x less memory *and* up to ~4x faster decoding**. That is why edge LLM work is mostly about quantization.

In [ ]:
from edge_vlm.memory import KNOWN_MODELS, budget_rows

nano = PROFILES['orin_nano_8gb']
print(format_table(budget_rows(KNOWN_MODELS.values(), [nano], ['fp16', 'int8', 'int4'])))

**How to read this:**

* SmolVLM-256M/500M fit easily even in FP16, so they are good first targets.
* 2-3B VLMs fit on the Orin Nano only when quantized (INT8/INT4), and decode at tens of tokens/s at best.
* 7B VLMs in FP16 do not fit on 8 GB. Use INT4 or a bigger module.
* `max_tok/s` is a roofline *upper bound*. Real numbers are typically 50-80% of it.

For our robot an action is ~5-15 tokens, so even 30 tok/s is acceptable for System 2. **Prefill**, which processes the image tokens plus the prompt, often dominates. That is why fewer image tokens (pixel shuffle, lower resolution) matter so much.

In [ ]:
from edge_vlm.memory import kv_cache_bytes, GB

# Why GQA matters: same 1.7B-style LM (24 layers, 32 heads x 64 dim), 4k-token context
for name, kv_heads in [('MHA (32 kv heads)', 32), ('GQA (8 kv heads)', 8), ('GQA (2 kv heads)', 2)]:
    print(f'{name:20s} KV cache = {kv_cache_bytes(24, kv_heads, 64, 4096) / GB:.3f} GB')

In [ ]:
from edge_vlm.memory import decode_tokens_per_sec_upper_bound, BYTES_PER_PARAM

# Decode speed bound vs LM size and precision on every Jetson
rows = []
for lm_params in [135e6, 360e6, 1.7e9, 3e9]:
    for prec in ['fp16', 'int8', 'int4']:
        row = {'LM params': f'{lm_params/1e9:.2f}B', 'precision': prec}
        for p in PROFILES.values():
            row[p.key] = round(decode_tokens_per_sec_upper_bound(lm_params * BYTES_PER_PARAM[prec], p.memory_bandwidth_gbs))
        rows.append(row)
print('max tokens/s (batch 1, memory-bound roofline)')
print(format_table(rows))

## Latency budget for the fetch robot

| Component | Rate | Budget per call | Runs on |
|---|---|---|---|
| Motor control / odometry | 100-1000 Hz | < 1 ms | MCU or CPU |
| Obstacle avoidance / path following (Nav2) | 10-20 Hz | < 50 ms | CPU (+GPU costmaps) |
| Localization (cuVSLAM) | 30 Hz | < 30 ms | GPU |
| **VLM skill selection (ours)** | **1-5 Hz** | **200-1000 ms** | **GPU** |
| Speech-to-text | on demand | < 1 s | GPU/CPU |

The VLM shares the GPU with cuVSLAM and the costmaps. Measure everything together on the real device (notebook 06).

## Learning path (and why in this order)

1. **01**: understand every tensor in a VLM by building a tiny one (2.4M params).
2. **02-03**: train it on a simulated fetch task, so it learns to look at images to decide `pick` vs `navigate`.
3. **04**: do the same with a *pretrained* small VLM (SmolVLM-256M), which understands free-form language.
4. **05**: make it small and fast (quantization, KV cache, token reduction, ONNX).
5. **06**: run it on a Jetson with TensorRT and measure memory, latency and power.
6. **07**: close the loop and map each piece onto ROS 2 / Isaac ROS for the real robot.

### Exercises
* Change `reserved_gb` for the Orin Nano to 3.5 (desktop GUI + ROS + RealSense). Which models still fit in FP16?
* Add a `ModelSpec` for a VLM you are curious about (look up its `config.json` on Hugging Face) and budget it.